In [1]:
import os
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset, Subset
import torch
from torch import nn
from torch.optim import lr_scheduler
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import time



In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [ ]:
img_size = 224



train_dir = r"C:\Users\dmin\HUST\20241\DeepLearning\New46Classes\train"
test_dir = r"C:\Users\dmin\HUST\20241\DeepLearning\New46Classes\val"


base_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor()
])

rotate_90_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomRotation(degrees=(90, 90)),
    transforms.ToTensor()
])

rotate_180_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomRotation(degrees=(180, 180)),
    transforms.ToTensor()
])

random_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),  
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop((img_size, img_size), scale=(0.8, 1.0)),  
    transforms.ToTensor(),
])

random_transform2 = transforms.Compose([
    transforms.Resize((img_size, img_size)),  
    transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.1, hue=0.2), 
    transforms.ToTensor(),
])

base_train_dataset = datasets.ImageFolder(root=train_dir, transform=base_transform)
targets = np.array([label for _, label in base_train_dataset])
num_classes = len(base_train_dataset.class_to_idx)

print(f'Training dataset has {len(base_train_dataset)} images')
print(f'Training dataset has {num_classes} labels')


rotated_90_train_dataset = datasets.ImageFolder(root=train_dir, transform=rotate_90_transform)
rotated_180_train_dataset = datasets.ImageFolder(root=train_dir, transform=rotate_180_transform)
random_train_dataset = datasets.ImageFolder(root=train_dir, transform=random_transform)
random2_train_dataset = datasets.ImageFolder(root=train_dir, transform=random_transform2)


train_dataset = ConcatDataset([base_train_dataset]) #doan nay muon them augmentation data thi thay = ConcatDataset([base_train_dataset, rotated_90_dataset,......])


val_dataset = datasets.ImageFolder(root=test_dir, transform=base_transform)


train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)


train_targets = [label for _, label in base_train_dataset]
class_weights = compute_class_weight('balanced', classes=np.unique(train_targets), y=train_targets)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Training samples (augmented): {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")




Training dataset has 6677 images
Training dataset has 46 labels
Training samples (augmented): 6677
Validation samples: 1670


In [4]:
class CNNClassification(nn.Module):
    def __init__(self,num_classes):
        super(CNNClassification, self).__init__()
        
        self.CNN_Model = nn.Sequential(
            # Block 1: Two Conv layers + Pooling
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),  # Output: 112x112x32

            # Block 2: Two Conv layers + Pooling
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),  # Output: 56x56x64

            # Block 3: Three Conv layers + Pooling
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),  # Output: 28x28x128

            # Block 4: Three Conv layers + Pooling
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),  # Output: 14x14x256

            # Block 5: Three Conv layers + Pooling with 256 filters
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),  # Output: 7x7x256


            nn.Flatten(),  
            nn.Dropout(0.3),
            nn.Linear(256 * 7 * 7, 2048),  
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes) 
        )

    def forward(self, x):
        return self.CNN_Model(x)


In [5]:
Cnn_model = CNNClassification(num_classes= num_classes)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.SGD(Cnn_model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0005)
Cnn_model.to(device)
total_parameters = sum(p.numel() for p in Cnn_model.parameters() if p.requires_grad)
print(f"Total number of parameters: {total_parameters}")
# Learning Rate Scheduler with ReduceLROnPlateau
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=30, verbose=True)

def clip_gradient(optimizer, grad_clip):
    for group in optimizer.param_groups:
        for param in group['params']:
            if param.grad is not None:
                param.grad.data.clamp_(-grad_clip, grad_clip)

Total number of parameters: 31521870


c:\Users\dmin\anaconda3\anaconda\envs\test_env\Lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [6]:
import copy

def test_the_model(model, test_dataloader):
    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0
    running_loss = 0.0

    # Disable gradient calculation during testing
    with torch.no_grad():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)

            
            loss = criterion(outputs, labels)
            #Get the top k predicted labels for each sample
            _, topk_preds = torch.topk(probabilities, k=1, dim=1)
            #Check if the true label is in the top k predicted labels
            correct += torch.sum(topk_preds.eq(labels.view(-1, 1))).item()
            total += labels.size(0)
            running_loss += loss.item()

    #accuracy of top k prediction
    accuracy = (correct / total) * 100
    avg_loss = running_loss / len(test_dataloader)
    return accuracy, avg_loss



def train_the_model(num_epochs=5, grad_clip=1.0):
    accuracies = []
    test_accuracies = []
    max_accuracy = 0
    best_model = None

    for epoch in range(num_epochs):
        Cnn_model.train() 
        correct = 0
        total = 0
        running_loss = 0.0
        start_epoch = time.time()

        for i, (images, labels) in enumerate(train_dataloader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = Cnn_model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            clip_gradient(optimizer, grad_clip)

            optimizer.step()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            running_loss += loss.item()

        time_complete_epoch = time.time() - start_epoch
        train_accuracy = (correct / total) * 100
        accuracies.append(train_accuracy)

        
        test_accuracy, test_loss = test_the_model(Cnn_model, val_dataloader)
        test_accuracies.append(test_accuracy)

        # Reduce learning rate based on validation loss
        scheduler.step(test_loss)

        
        if test_accuracy > max_accuracy:
            best_model = copy.deepcopy(Cnn_model)
            max_accuracy = test_accuracy
            print(f"Saving best model with Test Accuracy: {test_accuracy:.2f}%")
        
        print(f"Epoch {epoch + 1}/{num_epochs}, "
              f"Train Loss: {running_loss / len(train_dataloader):.4f}, "
              f"Train Accuracy: {train_accuracy:.2f}%, "
              f"Test Loss: {test_loss:.4f}, "
              f"Test Accuracy: {test_accuracy:.2f}%, "
              f"Time: {time_complete_epoch:.2f} seconds")

    #plot the accuracies
    plt.plot(accuracies, label='Train Accuracy')
    plt.plot(test_accuracies, label='Test Accuracy')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and Testing Accuracy Over Epochs")
    plt.legend()
    plt.show()

    return best_model

In [7]:
start_training_time = time.time()
result_model = train_the_model(400)
print(f'Training time: {(time.time() - start_training_time) / 60} minutes.')

Saving best model with Test Accuracy: 4.43%
Epoch 1/400, Train Loss: 3.8415, Train Accuracy: 2.71%, Test Loss: 3.7761, Test Accuracy: 4.43%, Time: 35.26 seconds
Saving best model with Test Accuracy: 8.14%
Epoch 2/400, Train Loss: 3.7207, Train Accuracy: 4.93%, Test Loss: 3.5786, Test Accuracy: 8.14%, Time: 33.88 seconds


KeyboardInterrupt: 

# Calculate the 20 worst F1-scores 

In [ ]:
from sklearn.metrics import f1_score

def calculate_f1_per_class(model, val_dataloader, num_classes, device):
    model.eval()  
    all_labels = []
    all_preds = []

    with torch.no_grad():  
        for images, labels in val_dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    

    f1_scores = f1_score(all_labels, all_preds, average=None, labels=np.arange(num_classes))
    
    return f1_scores

f1_scores = calculate_f1_per_class(result_model, val_dataloader, num_classes=num_classes, device=device)


lowest_f1_classes = np.argsort(f1_scores)[:20]
lowest_f1_scores = f1_scores[lowest_f1_classes]

class_to_idx = base_train_dataset.class_to_idx
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

for idx, class_idx in enumerate(lowest_f1_classes):
    print(f"Class {idx_to_class[class_idx],class_idx}: F1-score = {lowest_f1_scores[idx]:.4f}")

Class ('n02088466-bloodhound', 5): F1-score = 0.6462
Class ('n02100236-German_short-haired_pointer', 20): F1-score = 0.7143
Class ('n02088094-Afghan_hound', 4): F1-score = 0.7255
Class ('n02102177-Welsh_springer_spaniel', 23): F1-score = 0.7458
Class ('n02110958-pug', 35): F1-score = 0.7532
Class ('n02108422-bull_mastiff', 32): F1-score = 0.7606
Class ('n02089973-English_foxhound', 7): F1-score = 0.7719
Class ('n02086646-Blenheim_spaniel', 2): F1-score = 0.7778
Class ('n02089078-black-and-tan_coonhound', 6): F1-score = 0.7838
Class ('n02113978-Mexican_hairless', 43): F1-score = 0.7869
Class ('n02104365-schipperke', 25): F1-score = 0.7879
Class ('n02091467-Norwegian_elkhound', 9): F1-score = 0.7887
Class ('n02086910-papillon', 3): F1-score = 0.7895
Class ('n02105505-komondor', 26): F1-score = 0.7937
Class ('n02112018-Pomeranian', 38): F1-score = 0.7952
Class ('n02112706-Brabancon_griffon', 41): F1-score = 0.8000
Class ('n02091244-Ibizan_hound', 8): F1-score = 0.8046
Class ('n02111129-Le

In [ ]:
from sklearn.metrics import classification_report


def get_predictions(model, val_dataloader, device):
    model.eval()  
    all_labels = []
    all_preds = []
    
    with torch.no_grad():  
        for images, labels in val_dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    return all_labels, all_preds


true_labels, predicted_labels = get_predictions(result_model, val_dataloader, device)


report = classification_report(true_labels, predicted_labels, target_names= original_dataset.classes)
print(report)


                                       precision    recall  f1-score   support

           n02085782-Japanese_spaniel       0.94      0.89      0.92        37
                n02085936-Maltese_dog       0.89      0.92      0.90        51
           n02086646-Blenheim_spaniel       0.82      0.74      0.78        38
                   n02086910-papillon       0.81      0.77      0.79        39
               n02088094-Afghan_hound       0.69      0.77      0.73        48
                 n02088466-bloodhound       0.75      0.57      0.65        37
    n02089078-black-and-tan_coonhound       0.69      0.91      0.78        32
           n02089973-English_foxhound       0.85      0.71      0.77        31
               n02091244-Ibizan_hound       0.71      0.92      0.80        38
         n02091467-Norwegian_elkhound       0.88      0.72      0.79        39
         n02092002-Scottish_deerhound       0.82      0.89      0.86        47
                 n02092339-Weimaraner       0.90   

In [ ]:
#torch.save(result_model.state_dict(), 'DogBreed_83_41percent_top1_46classes_5-11-2024.pth')